# SPINE-GPE v7 — Fechamento da Fase 0 v1.0.1

Fluxo sequencial e fail-closed:

1. **RAIS Substantive Adjudication Lock**;
2. **Decodificação editorial dos perfis RAIS**;
3. **Master Harmonization & Evidence Lock**.

O notebook não relê a RAIS bruta. Ele usa os locks e o Parquet certificado dos vínculos ativos.

In [ ]:
from google.colab import drive, files
from pathlib import Path
import json
import shutil
import subprocess
import sys
import zipfile
import pandas as pd

drive.mount('/content/drive', force_remount=False)

ROOT = Path('/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7')
ROOT.mkdir(parents=True, exist_ok=True)

ZIP_NAME = 'SPINE_GPEv7_PHASE0_CLOSURE_PACKAGE_v1.0.1.zip'
ZIP_PATH = Path('/content') / ZIP_NAME

if not ZIP_PATH.is_file():
    print(f'Envie agora o arquivo {ZIP_NAME}.')
    uploaded = files.upload()
    if ZIP_NAME not in uploaded:
        raise FileNotFoundError(f'Arquivo esperado: {ZIP_NAME}')

print('Pacote:', ZIP_PATH)
print('Tamanho MB:', round(ZIP_PATH.stat().st_size / 1024**2, 2))

In [ ]:
INSTALL_DIR = ROOT / 'scripts' / 'phase0_closure_v101'
if INSTALL_DIR.exists():
    shutil.rmtree(INSTALL_DIR)
INSTALL_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH) as archive:
    for member in [name for name in archive.namelist() if not name.endswith('/')]:
        relative = Path(member)
        parts = relative.parts[1:] if len(relative.parts) > 1 else relative.parts
        if not parts:
            continue
        target = INSTALL_DIR.joinpath(*parts)
        target.parent.mkdir(parents=True, exist_ok=True)
        with archive.open(member) as src, target.open('wb') as dst:
            shutil.copyfileobj(src, dst)

ENGINE = INSTALL_DIR / 'SPINE_GPEv7_PHASE0_CLOSURE_v1.0.1.py'
CODEBOOK = INSTALL_DIR / 'rais_editorial_codebook_ptbr_v1.0.1.csv'
REQ = INSTALL_DIR / 'requirements_SPINE_GPEv7_PHASE0_CLOSURE_v1.0.1.txt'

assert ENGINE.is_file(), ENGINE
assert CODEBOOK.is_file(), CODEBOOK
assert REQ.is_file(), REQ
print('Instalado em:', INSTALL_DIR)

In [ ]:
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REQ)],
    check=True,
)
print('Dependências instaladas.')

## 1. Audit prévio

O audit confirma a presença dos locks upstream, do Parquet RAIS e do codebook editorial. Não cria os locks finais.

In [ ]:
AUDIT_RUN_ID = 'phase0_closure_audit_v101'
cmd_audit = [
    sys.executable, str(ENGINE),
    '--root', str(ROOT),
    '--mode', 'audit',
    '--stage', 'all',
    '--run-id', AUDIT_RUN_ID,
    '--codebook', str(CODEBOOK),
    '--strict',
]

print('Executando audit:')
print(' '.join(cmd_audit))
print()
process = subprocess.Popen(
    cmd_audit,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert process.stdout is not None
for line in process.stdout:
    print(line, end='')
audit_exit_code = process.wait()
print()
print('Audit exit code:', audit_exit_code)

In [ ]:
AUDIT_LOCK = ROOT / '00_admin' / 'PHASE0_CLOSURE_AUDIT_LOCK.json'
assert AUDIT_LOCK.is_file(), AUDIT_LOCK
audit_lock = json.loads(AUDIT_LOCK.read_text(encoding='utf-8'))
print(json.dumps(audit_lock, ensure_ascii=False, indent=2))
assert audit_exit_code == 0
assert audit_lock['status'] == 'AUDIT_PASSED'
assert audit_lock['critical_failures'] == []
print()
print('PHASE 0 CLOSURE AUDIT PASSED')

## 2. Execução completa

Esta célula produz os três locks, seus freezes e as tabelas finais da Fase 0.

In [ ]:
FULL_RUN_ID = 'phase0_closure_final_v101'
cmd_full = [
    sys.executable, str(ENGINE),
    '--root', str(ROOT),
    '--mode', 'full',
    '--stage', 'all',
    '--run-id', FULL_RUN_ID,
    '--codebook', str(CODEBOOK),
    '--strict',
]

print('Executando full:')
print(' '.join(cmd_full))
print()
process = subprocess.Popen(
    cmd_full,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert process.stdout is not None
for line in process.stdout:
    print(line, end='')
full_exit_code = process.wait()
print()
print('Full exit code:', full_exit_code)

In [ ]:
ADMIN = ROOT / '00_admin'
LOCKS = {
    'rais_adjudication': ADMIN / 'RAIS_SUBSTANTIVE_ADJUDICATION_LOCK.json',
    'rais_editorial': ADMIN / 'RAIS_EDITORIAL_PROFILE_LOCK.json',
    'phase0_master': ADMIN / 'SPINE_GPE_PHASE0_MASTER_LOCK.json',
}

payloads = {}
for name, path in LOCKS.items():
    assert path.is_file(), path
    payloads[name] = json.loads(path.read_text(encoding='utf-8'))
    print(name, '=>', payloads[name].get('status'))

assert full_exit_code == 0
assert payloads['rais_adjudication']['status'] == 'ADJUDICATED'
assert payloads['rais_editorial']['status'] == 'EDITORIAL_CERTIFIED'
assert payloads['phase0_master']['status'] == 'PHASE0_CERTIFIED'
assert payloads['phase0_master']['critical_failures'] == []

print()
print('FASE 0 CERTIFICADA E CONGELADA')

## 3. Revisão dos outputs finais

In [ ]:
master = payloads['phase0_master']
rais_adj = payloads['rais_adjudication']
rais_ed = payloads['rais_editorial']

income = pd.read_csv(rais_adj['artifacts']['income_adjudication'])
editorial = pd.read_csv(rais_ed['artifacts']['editorial_profile'])
evidence = pd.read_csv(master['artifacts']['evidence_matrix'])
rules = pd.read_csv(master['artifacts']['comparison_rules'])
claims = pd.read_csv(master['artifacts']['claim_registry'])

print('RAIS — adjudicação remuneratória')
display(income)

print('RAIS — perfil editorial')
display(editorial.head(200))

print('Matriz de evidência')
display(evidence)

print('Regras de comparação')
display(rules)

print('Claim registry')
display(claims)

In [ ]:
MASTER_FREEZE = ADMIN / 'SPINE_GPE_PHASE0_MASTER_FREEZE.json'
assert MASTER_FREEZE.is_file(), MASTER_FREEZE
freeze = json.loads(MASTER_FREEZE.read_text(encoding='utf-8'))
print(json.dumps(freeze, ensure_ascii=False, indent=2))
assert freeze['status'] == 'FROZEN'
assert freeze['read_only'] is True
print()
print('PRÓXIMA FASE:', freeze['next_phase'])